In [ ]:
!pip install pdfplumber chromadb sentence-transformers tqdm


In [ ]:
pip install chromadb langchain sentence-transformers transformers accelerate bitsandbytes huggingface_hub pypdf

In [ ]:
import pdfplumber
import chromadb
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

#Upload PCI-DSS PDF

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose your PCI-DSS PDF from your device
PDF_PATH = list(uploaded.keys())[0]
print("Using:", PDF_PATH)


Saving PCI-DSS-v4.0.1.pdf to PCI-DSS-v4.0.1 (1).pdf
Using: PCI-DSS-v4.0.1 (1).pdf


##Parse and chunk the PDF

In [ ]:
def extract_text_from_pdf(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for p in pdf.pages:
            text = p.extract_text()
            if text:
                pages.append(text)
    return "\n\n".join(pages)

full_text = extract_text_from_pdf(PDF_PATH)
print("Extracted characters:", len(full_text))

Extracted characters: 794888


In [ ]:
def chunk_text(text, chunk_size=800, overlap=200):
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, current = [], ""
    for p in paragraphs:
        if len(current) + len(p) + 2 <= chunk_size:
            current = current + "\n\n" + p if current else p
        else:
            if current: chunks.append(current)
            if len(p) <= chunk_size:
                current = p
            else:
                for i in range(0, len(p), chunk_size - overlap):
                    chunks.append(p[i:i+chunk_size])
                current = ""
    if current:
        chunks.append(current)
    return chunks

chunks = chunk_text(full_text, 800, 200)
print("Chunks:", len(chunks))

Chunks: 1512


##Generate embeddings (local model just baseline )

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm import tqdm

model = SentenceTransformer("all-MiniLM-L12-v2")
texts = chunks
embeddings = []
for i in tqdm(range(0, len(texts), 64)):
    batch = texts[i:i+64]
    emb = model.encode(batch, convert_to_numpy=True)
    embeddings.append(emb)
embeddings = np.vstack(embeddings)
print("Embeddings shape:", embeddings.shape)


100%|██████████| 24/24 [00:03<00:00,  6.83it/s]

Embeddings shape: (1512, 384)


In [ ]:
pip show chromadb


Name: chromadb
Version: 1.2.2
Summary: Chroma.
Home-page: https://github.com/chroma-core/chroma
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, posthog, pybase64, pydantic, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: 


**Create & store in ChromaDb**

In [ ]:
client = chromadb.PersistentClient(path="./chroma_pci_dss")

collection_name = "pci_dss_v4"
try:
    client.delete_collection(collection_name)
except:
    pass

collection = client.create_collection(name=collection_name)

ids = [f"chunk_{i}" for i in range(len(chunks))]

collection.add(
    ids=ids,
    documents=chunks,
    embeddings=embeddings.tolist(),
    metadatas=[{"chunk_id": i} for i in range(len(chunks))]
)

print("Collection created with", collection.count(), "documents.")

Collection created with 1512 documents.


**Test semantic query**

In [ ]:
query = "What are the main PCI DSS encryption requirements?"
query_emb = model.encode([query], convert_to_numpy=True)

results = collection.query(
    query_embeddings=query_emb,
    n_results=3
)

print("\n Query:", query)
for doc in results["documents"][0]:
    print("—", doc[:300], "...\n")



 Query: What are the main PCI DSS encryption requirements?
— nts.
Table 1. Principal PCI DSS Requirements
PCI Data Security Standard – High Level Overview
Build and Maintain a Secure Network and Systems 1. Install and Maintain Network Security Controls.
2. Apply Secure Configurations to All System Components.
Protect Account Data 3. Protect Stored Account Dat ...

— according to
PCI DSS.
The following are each in scope for PCI DSS:
 Systems performing encryption and/or decryption of cardholder data, and systems performing key management functions,
 Encrypted cardholder data that is not isolated from the encryption and decryption and key management processes,
 ...

— 4 Scope of PCI DSS Requirements
PCI DSS requirements apply to:
 The cardholder data environment (CDE), which is comprised of:
– System components, people, and processes that store, process, or transmit cardholder data and/or sensitive authentication data,
and,
– System components that may not store ...



## Load Mistral-7B-Instruct-v0.3

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [ ]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L12-v2")

In [ ]:

#  Load Mistral-7B-Instruct-v0.3


model_name = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("✅ Mistral model loaded successfully.")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Mistral model loaded successfully.


## Define Retrieval-Augmented QA

In [ ]:
def retrieve_context(query, k=3):
    """Retrieve top-k relevant chunks from Chroma."""
    query_emb = embedder.encode([query])
    results = collection.query(
        query_embeddings=query_emb.tolist(),
        n_results=k
    )
    contexts = results["documents"][0]
    return "\n".join(contexts)

In [ ]:
def generate_answer(query, context):
    """Generate an answer using Mistral."""
    prompt = f"""You are a PCI-DSS v4.0 compliance assistant.
Use the provided context to answer the user's question accurately.

Context:
{context}

Question:
{query}

Answer:"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=300)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

## examples queires

In [ ]:
query_1 = "What are the PCI DSS requirements for storing cardholder data?"
context_1 = retrieve_context(query_1)
answer_1 = generate_answer(query_1, context_1)

print("\n Query:", query_1)
print("\n Answer:\n", answer_1)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



🔍 Query: What are the PCI DSS requirements for storing cardholder data?

🧠 Answer:
 You are a PCI-DSS v4.0 compliance assistant.
Use the provided context to answer the user's question accurately.

Context:
4 Scope of PCI DSS Requirements
PCI DSS requirements apply to:
 The cardholder data environment (CDE), which is comprised of:
– System components, people, and processes that store, process, or transmit cardholder data and/or sensitive authentication data,
and,
– System components that may not store, process, or transmit CHD/SAD but have unrestricted connectivity to system components
that store, process, or transmit CHD/SAD.
AND
 System components, people, and processes that could impact the security of cardholder data and/or sensitive authentication data.4
“System components” include network devices, servers, computing devices, virtual components, cloud components, and software. Examples
of system components include but are not limited to:
 Systems that store, process, or transmi

## SQL Compliance Check

In [ ]:
def check_sql_compliance(sql_query: str):
    """Check if a SQL query complies with PCI DSS."""
    question = f"Does the following SQL query comply with PCI DSS v4.0 security and data handling standards?\n\n{sql_query}"
    context = retrieve_context("PCI DSS data storage and encryption requirements")
    answer = generate_answer(question, context)
    return answer

In [ ]:
sql_example = """
SELECT customer_name, card_number
FROM payments
WHERE card_number LIKE '4%';
"""

print("\n💾 SQL Compliance Check:\n", check_sql_compliance(sql_example))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



💾 SQL Compliance Check:
 You are a PCI-DSS v4.0 compliance assistant.
Use the provided context to answer the user's question accurately.

Context:
nts.
Table 1. Principal PCI DSS Requirements
PCI Data Security Standard – High Level Overview
Build and Maintain a Secure Network and Systems 1. Install and Maintain Network Security Controls.
2. Apply Secure Configurations to All System Components.
Protect Account Data 3. Protect Stored Account Data.
4. Protect Cardholder Data with Strong Cryptography During Transmission Over Open,
Public Networks.
Maintain a Vulnerability Management Program 5. Protect All Systems and Networks from Malicious Software.
6. Develop and Maintain Secure Systems and Software.
Implement Strong Access Control Measures 7. Restrict Access to System Components and Cardholder Data by Business Need to Know.
8. Identify Users and Authenticate Access to System Components.
9. Restrict Physical Access to Cardholder Data.

thods. Full disk encryption helps to protect data i

In [ ]:
sql_1 = "SELECT cardholder_name, email FROM customers WHERE id=123;"
sql_2 = "SELECT * FROM transactions;"

check_sql_compliance(sql_1)



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


"You are a PCI-DSS v4.0 compliance assistant.\nUse the provided context to answer the user's question accurately.\n\nContext:\nnts.\nTable 1. Principal PCI DSS Requirements\nPCI Data Security Standard – High Level Overview\nBuild and Maintain a Secure Network and Systems 1. Install and Maintain Network Security Controls.\n2. Apply Secure Configurations to All System Components.\nProtect Account Data 3. Protect Stored Account Data.\n4. Protect Cardholder Data with Strong Cryptography During Transmission Over Open,\nPublic Networks.\nMaintain a Vulnerability Management Program 5. Protect All Systems and Networks from Malicious Software.\n6. Develop and Maintain Secure Systems and Software.\nImplement Strong Access Control Measures 7. Restrict Access to System Components and Cardholder Data by Business Need to Know.\n8. Identify Users and Authenticate Access to System Components.\n9. Restrict Physical Access to Cardholder Data.\n\nthods. Full disk encryption helps to protect data in the e